# 비동기 라우팅과 Pydantic 스키마

요청 본문을 타입으로 선언하면 정상 입력과 잘못된 입력이 어떻게 나뉘는지 확인합니다.

## 실행 환경 준비

현재 폴더나 상위 폴더에서 `labs/part7/ch30-fastapi-serving`을 찾아 작업 폴더로 사용합니다. 비밀 키를 출력하지 않습니다.

In [1]:
import os
from pathlib import Path
start = Path.cwd().resolve()
lab = next((p for p in (start, *start.parents)
    if (p / 'agent_service').is_dir()), None)
assert lab is not None, 'agent_service가 있는 Ch30 실습 폴더에서 실행하세요.'
os.chdir(lab)
print('실습 폴더:', Path.cwd().name)

실습 폴더: ch30-fastapi-serving


## 기본 실행

작은 FastAPI 앱에 `async def` 라우트와 Pydantic 요청 모델을 연결합니다.

In [2]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

class Question(BaseModel):
    text: str = Field(min_length=1, max_length=20)

mini = FastAPI()
@mini.post('/ask')
async def ask(body: Question):
    return {'length': len(body.text)}

client = TestClient(mini)

## 정상 요청 관찰

JSON 본문이 `Question` 객체로 변환된 뒤 길이가 반환됩니다.

In [3]:
ok = client.post('/ask', json={'text': '안녕하세요'})
print(ok.status_code, ok.json())
assert ok.json() == {'length': 5}

200 {'length': 5}


## 실패 재현

빈 문자열은 응용 코드에 도달하기 전에 스키마 검증에서 거부됩니다.

In [4]:
bad = client.post('/ask', json={'text': ''})
print(bad.status_code, bad.json()['detail'][0]['type'])
assert bad.status_code == 422

422 string_too_short


## 수정

같은 요청에서 내용을 한 글자 이상 채우면 검증을 통과합니다.

In [5]:
fixed = client.post('/ask', json={'text': '확인'})
print(fixed.status_code, fixed.json())
assert fixed.status_code == 200

200 {'length': 2}
